In [1]:
import unicodedata
import string 
import re 
import torch

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#print(device)

In [3]:
SOS_token = 0
EOS_token = 1
MAX_LENGTH = 20

In [4]:
class Lang:
    """Class to store vocabulary and word to index mappings for a language"""
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1:"EOS"}
        self.n_words = 2

    def addSentece(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words +=1
        else:
            self.word2count[word] += 1

In [5]:
def unicodeToAscii(s):
    """Turn a Unicode string to plain ASCII."""
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn')

def normalizeString(s):
    """Lowercase, trim, and remove non-letter characters."""
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s.strip()

In [7]:
def readLangs(lang1, lang2, reverse=False):
    """Read the text file and split into translational pairs."""
    print("Reading Lines""")
    # Split every line into pairs and normalize
    # eng-fra.txt is formatted as: English \t French
    lines = open('data/eng-fra.txt',encoding='utf-8').read().strip().split('\n')
    pairs =[normalizeString(s) for s in l.split('\t')[:2] for l in lines]

    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)

    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs

def filterPair(p):
    """Filter to keep all sentences under MAX_LENGTH, ignoring prefixes."""
    return len(p[0].split(' ')) < MAX_LENGTH and \
           len(p[1].split(' ')) < MAX_LENGTH
    
def filterPairs(pairs):
    return [p for p in pairs if filterPair(p)]

In [8]:
def prepareData(lang1, lang2, reverse = False):
    """Read the text file and split into transaltional pairs."""
    print("Reading lines")
    
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)

    print(f"Read {len(pairs)} sentence pairs")
    
    pairs = filterPairs(pairs)
    print(f"Trimmed to {len(pairs)} sentence pairs")

    print("Counting words")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])

    print("counted words")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype= torch.long, device=device).view(-1,1)

def tensorsFromPair(pair, input_lang, output_lang):
    """Convert a [English, French] string pair into a pair of tensors."""
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)
        

SyntaxError: invalid syntax (3616943195.py, line 1)